# Extract Text from Product Documentation PDFs
This notebook extracts text from PDF files in a Databricks volume and stores them in a Unity Catalog table.

**Configuration:**
- Source: `/Volumes/llmagent/dev/data_volume/01_Data_Files/product_docs/`
- Target: `llmagent.dev.product_details`
- Batch Size: 50 files (to manage memory for ~500 files)

**Schema:**
- `product_name` - PDF filename without extension
- `product_doc` - Extracted text content

## 1. Setup and Configuration

In [0]:
%pip install PyPDF2

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
import os
import tempfile
import PyPDF2
from datetime import datetime

# Initialize Spark Session
spark = SparkSession.builder.appName("PDFTextExtraction").getOrCreate()

# Configuration
VOLUME_PATH = "/Volumes/llmagent/dev/data_volume/01_Data_Files/product_docs"
CATALOG = "llmagent"
SCHEMA = "dev"
TABLE_NAME = "product_details"
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"
BATCH_SIZE = 50  # Process 50 PDFs at a time to manage memory

print(f"Configuration:")
print(f"  Volume Path: {VOLUME_PATH}")
print(f"  Target Table: {FULL_TABLE_NAME}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Define Helper Functions

In [0]:
def extract_text_from_pdf(file_path: str) -> str:
    """
    Extract text from a PDF file using PyPDF2.
    
    Args:
        file_path: Path to the PDF file
    
    Returns:
        Extracted text content
    """
    try:
        text_content = []
        with open(file_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page_num in range(len(pdf_reader.pages)):
                page = pdf_reader.pages[page_num]
                text = page.extract_text()
                if text:
                    text_content.append(text)
        return "\n".join(text_content)
    except Exception as e:
        print(f"Error extracting text from {file_path}: {str(e)}")
        return ""

def get_product_name(filename: str) -> str:
    """Extract product name from filename (without extension)."""
    return os.path.splitext(filename)[0]

def process_pdfs_batch(batch_files: list) -> list:
    """
    Process a batch of PDF files and extract text.
    
    Args:
        batch_files: List of file info objects for the batch
    
    Returns:
        List of tuples (product_name, product_doc)
    """
    batch_data = []

    for file_info in batch_files:
        file_path = file_info.path
        filename = file_info.name

        # Create temporary local copy and extract text
        with tempfile.NamedTemporaryFile(suffix='.pdf', delete=False) as tmp_file:
            try:
                # Copy file from volume to temporary location
                dbutils.fs.cp(file_path, f"file:{tmp_file.name}", recurse=False)

                # Extract text
                text_content = extract_text_from_pdf(tmp_file.name)
                product_name = get_product_name(filename)

                if text_content.strip():
                    batch_data.append((product_name, text_content))
                    print(f"  ✓ {filename}: {len(text_content)} chars")
                else:
                    print(f"  ⚠ {filename}: No text extracted")

            except Exception as e:
                print(f"  ✗ {filename}: {str(e)}")
            finally:
                # Clean up temporary file
                try:
                    os.unlink(tmp_file.name)
                except:
                    pass

    return batch_data

def create_table_if_not_exists():
    """Create the target table if it doesn't exist."""
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {FULL_TABLE_NAME} (
            product_name STRING,
            product_doc STRING
        )
        USING DELTA
    """)
    print(f"✓ Table {FULL_TABLE_NAME} ready")

print("✓ Helper functions defined")

## 3. Create Target Table

In [0]:
create_table_if_not_exists()

## 4. List and Count PDFs

In [0]:
# Check if volume path exists and list PDFs
try:
    files = dbutils.fs.ls(VOLUME_PATH)
except Exception as e:
    print(f"Error accessing volume path: {str(e)}")
    raise

# Filter for PDF files only
pdf_files = [f for f in files if f.name.lower().endswith('.pdf')]

if not pdf_files:
    print(f"No PDF files found in {VOLUME_PATH}")
else:
    print(f"✓ Found {len(pdf_files)} PDF files")
    print(f"\nFirst 10 PDFs:")
    for i, pdf in enumerate(pdf_files[:10], 1):
        print(f"  {i}. {pdf.name}")

## 5. Process PDFs in Batches

In [0]:
total_files = len(pdf_files)
print(f"Processing {total_files} PDF files in batches of {BATCH_SIZE}\n")

# Process files in batches
total_processed = 0
total_succeeded = 0

for batch_start in range(0, total_files, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, total_files)
    batch_files = pdf_files[batch_start:batch_end]
    batch_num = (batch_start // BATCH_SIZE) + 1
    total_batches = (total_files + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"\n{'='*70}")
    print(f"Batch {batch_num}/{total_batches} (Files {batch_start + 1}-{batch_end})")
    print(f"{'='*70}")

    # Process batch
    batch_data = process_pdfs_batch(batch_files)
    total_processed += len(batch_files)
    total_succeeded += len(batch_data)

    if batch_data:
        # Create DataFrame from batch
        schema = StructType([
            StructField("product_name", StringType(), True),
            StructField("product_doc", StringType(), True)
        ])

        df_batch = spark.createDataFrame(batch_data, schema=schema)

        # Append to table (using overwrite for first batch, append for rest)
        mode = "overwrite" if batch_start == 0 else "append"
        df_batch.write.mode(mode).option("mergeSchema", "true").saveAsTable(FULL_TABLE_NAME)

        print(f"\n✓ Batch written ({len(batch_data)} rows)")
    else:
        print(f"\n⚠ No data to write from this batch")

print(f"\n{'='*70}")
print("Processing Complete!")
print(f"{'='*70}")
print(f"Total files processed: {total_processed}")
print(f"Successfully extracted: {total_succeeded}")
print(f"Failed: {total_processed - total_succeeded}")

## 6. Verify Results

In [0]:
# Get final row count
row_count = spark.sql(f"SELECT COUNT(*) as count FROM {FULL_TABLE_NAME}").collect()[0]['count']
print(f"\nFinal table row count: {row_count}")

## 7. Sample Data

In [0]:
# Show sample data
display(spark.sql(f"""
    SELECT
        product_name,
        LENGTH(product_doc) as text_length,
        SUBSTR(product_doc, 1, 200) as preview
    FROM {FULL_TABLE_NAME}
    LIMIT 5
"""))

## 8. Table Statistics

In [0]:
# Show table statistics
stats = spark.sql(f"""
    SELECT
        COUNT(*) as total_records,
        MIN(LENGTH(product_doc)) as min_text_length,
        MAX(LENGTH(product_doc)) as max_text_length,
        AVG(LENGTH(product_doc)) as avg_text_length
    FROM {FULL_TABLE_NAME}
""")

display(stats)